In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3, os, glob
from tqdm import tqdm
import re
from IPython.display import clear_output

## Looking at the data in the training set

In [7]:
# the train folder contains two jsons in the root, loading and checking em out
import json 

with open(r"../Data/train/train.json", mode='r',encoding='utf-8') as r:
    train_json = json.load(r)

with open(r"../Data/train/train_tables.json", mode='r',encoding='utf-8') as r:
    train_tables_json = json.load(r)

In [6]:
train_json

[{'db_id': 'movie_platform',
  'question': 'Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.',
  'evidence': 'released in the year 1945 refers to movie_release_year = 1945;',
  'SQL': 'SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1'},
 {'db_id': 'movie_platform',
  'question': 'State the most popular movie? When was it released and who is the director for the movie?',
  'evidence': 'most popular movie refers to MAX(movie_popularity); when it was released refers to movie_release_year; director for the movie refers to director_name;',
  'SQL': 'SELECT movie_title, movie_release_year, director_name FROM movies ORDER BY movie_popularity DESC LIMIT 1 '},
 {'db_id': 'movie_platform',
  'question': 'What is the name of the longest movie title? When was it released?',
  'evidence': 'longest movie title refers to MAX(LENGTH(movie_title)); when it was released refers to movie_release_year

In [13]:
len(train_json)

9428

In [12]:
train_tables_json

[{'db_id': 'european_football_1',
  'table_names_original': ['divisions', 'matchs'],
  'table_names': ['divisions', 'matchs'],
  'column_names_original': [[-1, '*'],
   [0, 'division'],
   [0, 'name'],
   [0, 'country'],
   [1, 'Div'],
   [1, 'Date'],
   [1, 'HomeTeam'],
   [1, 'AwayTeam'],
   [1, 'FTHG'],
   [1, 'FTAG'],
   [1, 'FTR'],
   [1, 'season']],
  'column_names': [[-1, '*'],
   [0, 'division'],
   [0, 'name'],
   [0, 'country'],
   [1, 'Division'],
   [1, 'Date'],
   [1, 'HomeTeam'],
   [1, 'AwayTeam'],
   [1, 'Final-time Home-team Goals'],
   [1, 'Final-time Away-team Goals'],
   [1, 'Final-time Results'],
   [1, 'season']],
  'column_types': ['text',
   'text',
   'text',
   'text',
   'text',
   'date',
   'text',
   'text',
   'integer',
   'integer',
   'text',
   'integer'],
  'primary_keys': [1],
  'foreign_keys': [[4, 1]]},
 {'db_id': 'sales_in_weather',
  'table_names_original': ['sales_in_weather', 'weather', 'relation'],
  'table_names': ['sales_in_weather', 'weath

In [11]:
len(train_tables_json)

69

In [19]:
# listing the csvs in one of the database to see what it contains
db_descript_path = "../Data/train/train_databases/books/database_description"
text_descriptions = os.listdir("../Data/train/train_databases/books/database_description")

In [17]:
text_descriptions

['order_status.csv',
 'author.csv',
 'publisher.csv',
 'order_history.csv',
 'book_language.csv',
 'customer.csv',
 'order_line.csv',
 'book_author.csv',
 'address.csv',
 'address_status.csv',
 'country.csv',
 'customer_address.csv',
 'cust_order.csv',
 'book.csv',
 'shipping_method.csv']

In [33]:
display(pd.read_csv(db_descript_path+"/"+text_descriptions[0]))

,original_column_name,column_name,column_description,data_format,value_description
0,status_id,status id,the unique identifier of the order status,integer,NaN
1,status_value,status value,the status value,text,commonsense evidence:\nThe order statuses incl...


In [ ]:
ROOT = "../Data/train/train_databases"   # adjust
# Extracting the database from each file
dbs = {os.path.basename(p): glob.glob(os.path.join(p, "*.sqlite"))[0]
       for p in glob.glob(os.path.join(ROOT, "*")) if os.path.isdir(p)}
len(dbs), list(dbs)[:10]

(69,
 ['european_football_1',
  'sales_in_weather',
  'craftbeer',
  'soccer_2016',
  'restaurant',
  'movie',
  'olympics',
  'language_corpus',
  'app_store',
  'sales'])

In [ ]:
# for non db wizards, this is just a basic get info of the databases function
def schema(path):
    con = sqlite3.connect(path)
    tabs = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", con)["name"]
    out = {}
    for t in tabs:
        cols = pd.read_sql(f'PRAGMA table_info("{t}")', con)[["name","type","pk"]]
        fks  = pd.read_sql(f'PRAGMA foreign_key_list("{t}")', con)
        n    = pd.read_sql(f'SELECT COUNT(*) c FROM "{t}"', con)["c"][0]
        out[t] = {"rows": n, "cols": cols, "fks": fks[["from","table","to"]] if len(fks) else None}
    con.close()
    return out

s = schema(dbs["movie_platform"])
for t, d in s.items():
    print(f"\n=== {t}  ({d['rows']} rows)")
    display(d["cols"])
    if d["fks"] is not None: display(d["fks"])


=== lists  (79565 rows)


,name,type,pk
0,user_id,INTEGER,0
1,list_id,INTEGER,1
2,list_title,TEXT,0
3,list_movie_number,INTEGER,0
4,list_update_timestamp_utc,TEXT,0
5,list_creation_timestamp_utc,TEXT,0
6,list_followers,INTEGER,0
7,list_url,TEXT,0
8,list_comments,INTEGER,0
9,list_description,TEXT,0


,from,table,to
0,user_id,lists_users,user_id



=== movies  (226087 rows)


,name,type,pk
0,movie_id,INTEGER,1
1,movie_title,TEXT,0
2,movie_release_year,INTEGER,0
3,movie_url,TEXT,0
4,movie_title_language,TEXT,0
5,movie_popularity,INTEGER,0
6,movie_image_url,TEXT,0
7,director_id,TEXT,0
8,director_name,TEXT,0
9,director_url,TEXT,0



=== ratings_users  (4297641 rows)


... (4 outputs left out)

,from,table,to
0,user_id,lists,user_id
1,list_id,lists,list_id



=== ratings  (15517252 rows)


,name,type,pk
0,movie_id,INTEGER,0
1,rating_id,INTEGER,0
2,rating_url,TEXT,0
3,rating_score,INTEGER,0
4,rating_timestamp_utc,TEXT,0
5,critic,TEXT,0
6,critic_likes,INTEGER,0
7,critic_comments,INTEGER,0
8,user_id,INTEGER,0
9,user_trialist,INTEGER,0


,from,table,to
0,user_id,ratings_users,user_id
1,rating_id,ratings,rating_id
2,user_id,lists_users,user_id
3,movie_id,movies,movie_id


In [35]:
def q(db, sql, n=None):
    con = sqlite3.connect(dbs[db] if db in dbs else db)
    df = pd.read_sql(sql, con); con.close()
    return df.head(n) if n else df

q("movie_platform", "SELECT * FROM movies LIMIT 5")

,movie_id,movie_title,movie_release_year,movie_url,movie_title_language,movie_popularity,movie_image_url,director_id,director_name,director_url
0,1,La Antena,2007,http://mubi.com/films/la-antena,en,105,https://images.mubicdn.net/images/film/1/cache...,131,Esteban Sapir,http://mubi.com/cast/esteban-sapir
1,2,Elementary Particles,2006,http://mubi.com/films/elementary-particles,en,23,https://images.mubicdn.net/images/film/2/cache...,73,Oskar Roehler,http://mubi.com/cast/oskar-roehler
2,3,It's Winter,2006,http://mubi.com/films/its-winter,en,21,https://images.mubicdn.net/images/film/3/cache...,82,Rafi Pitts,http://mubi.com/cast/rafi-pitts
3,4,Kirikou and the Wild Beasts,2005,http://mubi.com/films/kirikou-and-the-wild-beasts,en,46,https://images.mubicdn.net/images/film/4/cache...,"89, 90","Michel Ocelot, Bénédicte Galup","http://mubi.com/cast/michel-ocelot, http://mub..."
4,5,Padre Nuestro,2007,http://mubi.com/films/padre-nuestro,en,7,https://images.mubicdn.net/images/film/5/cache...,92,Christopher Zalla,http://mubi.com/cast/christopher-zalla


In [ ]:
train = json.load(open("../Data/train/train.json"))
tables = {t["db_id"]: t for t in json.load(open("../Data/train/train_tables.json"))}

# showing one example
ex = train[0]
print(ex["question"]); print(ex["evidence"]); print(ex["SQL"]); print(ex["db_id"])

# run the gold query
q(ex["db_id"], ex["SQL"])

Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.
released in the year 1945 refers to movie_release_year = 1945;
SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1
movie_platform


,movie_title
0,Brief Encounter


In [38]:
def descs(db):
    d = os.path.join(ROOT, db, "database_description")
    return {os.path.basename(f)[:-4]: pd.read_csv(f, encoding="latin-1")
            for f in glob.glob(os.path.join(d, "*.csv"))}

descs("movie_platform")["movies"]

,original_column_name,column_name,column_description,data_format,value_description
0,movie_id,NaN,ID related to the movie on Mubi,integer,NaN
1,movie_title,NaN,Name of the movie,text,NaN
2,movie_release_year,NaN,Release year of the movie,integer,NaN
3,movie_url,NaN,URL to the movie page on Mubi,text,NaN
4,movie_title_language,NaN,"By default, the title is in English.",text,Only contains one value which is 'en'
5,movie_popularity,NaN,Number of Mubi users who love this movie,integer,NaN
6,movie_image_url,NaN,Image URL to the movie on Mubi,text,NaN
7,director_id,NaN,ID related to the movie director on Mubi,text,NaN
8,director_name,NaN,Full Name of the movie director,text,NaN
9,director_url,NaN,URL to the movie director page on Mubi,text,NaN


In [39]:
rows = []
for name, p in dbs.items():
    con = sqlite3.connect(p)
    tabs = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", con)["name"]
    rows.append({"db": name, "n_tables": len(tabs),
                 "size_mb": round(os.path.getsize(p)/1e6, 1)})
    con.close()
pd.DataFrame(rows).sort_values("size_mb", ascending=False)

,db,n_tables,size_mb
22,bike_share_1,4,5209.4
51,codebase_comments,4,4497.8
62,donor,4,4497.8
32,movie_platform,5,4311.1
57,world_development_indicators,6,2451.7
...,...,...,...
30,shipping,5,0.1
52,retail_world,8,0.1
16,human_resources,3,0.0
18,computer_student,4,0.0


In [ ]:
def schema_str(db_id):
    con = sqlite3.connect(dbs[db_id])
    ddl = pd.read_sql(
        "SELECT sql FROM sqlite_master WHERE sql IS NOT NULL", con)["sql"]
    con.close()
    return "\n\n".join(ddl)

In [61]:
import re
payload = {r["id"]: r for r in json.load(open("payload_full.json"))}
results = json.load(open("results.json"))

def clean(sql):
    sql = re.sub(r"```sql|```", "", sql).strip()
    return sql.split(";")[0]

def run(db_id, sql):
    con = sqlite3.connect(dbs[db_id])
    try:    return set(map(tuple, con.execute(sql).fetchall())), None
    except Exception as e: return None, str(e)
    finally: con.close()

rows = []
c = 1
for r in results:
    clear_output(wait=True)
    print(f"Processing {c}/{len(results)}", end="\r")
    c += 1
    p = payload[r["id"]]
    pred = clean(r["sql"])
    pset, err = run(p["db_id"], pred)
    gset, _   = run(p["db_id"], p["gold"])
    rows.append({"id": r["id"], "db": p["db_id"], "pred": pred,
                 "correct": 0 if err else int(pset == gset)})

res = pd.DataFrame(rows)
print("Execution Accuracy:", res.correct.mean())

Execution Accuracy: 0.6650403054730589


In [ ]:
# payload keyed by id; make sure each entry carries the question.
# if payload_full.json doesn't have "question", rebuild it from train (see note below).
def sample_table(res, payload, n=5):
    corr  = res[res.correct == 1].sample(min(n, (res.correct==1).sum()), random_state=0)
    wrong = res[res.correct == 0].sample(min(n, (res.correct==0).sum()), random_state=0)
    tbl = pd.concat([corr, wrong]).reset_index(drop=True)
    tbl["question"] = tbl["id"].map(lambda i: train[i]["question"])
    tbl["gold"]     = tbl["id"].map(lambda i: train[i]["SQL"])
    return tbl[["id", "db", "correct", "question", "pred", "gold"]]

tbl1 = sample_table(res, payload)

pd.set_option("display.max_colwidth", None)
tbl1

,id,db,correct,question,pred,gold
0,737,movie,1,"Among the movies starring Tom Cruise, which one of them has the best quality?",SELECT T2.Title FROM characters AS T1 INNER JOIN movie AS T2 ON T1.MovieID = T2.MovieID INNER JOIN actor AS T3 ON T1.ActorID = T3.ActorID WHERE T3.Name = 'Tom Cruise' ORDER BY T2.Rating DESC LIMIT 1,SELECT T1.Title FROM movie AS T1 INNER JOIN characters AS T2 ON T1.MovieID = T2.MovieID INNER JOIN actor AS T3 ON T3.ActorID = T2.ActorID WHERE T3.Name = 'Tom Cruise' ORDER BY T1.Rating DESC LIMIT 1
1,1213,talkingdata,1,How many people over the age of 50 do not have HTC One M8 Eye phones?,SELECT COUNT(DISTINCT T1.device_id) FROM gender_age AS T1 INNER JOIN phone_brand_device_model2 AS T2 ON T1.device_id = T2.device_id WHERE T1.age > 50 AND T2.phone_brand != 'HTC' AND T2.device_model != 'One M8 Eye',SELECT COUNT(T1.device_id) FROM gender_age AS T1 INNER JOIN phone_brand_device_model2 AS T2 ON T1.device_id = T2.device_id WHERE T1.age > 50 AND T2.device_model != 'One M8 Eye' AND T2.phone_brand != 'HTC'
2,8007,university,1,What is the university ID of the university with the largest student staff ratio?,SELECT university_id FROM university_year ORDER BY student_staff_ratio DESC LIMIT 1,SELECT university_id FROM university_year ORDER BY student_staff_ratio DESC LIMIT 1
3,1676,restaurant,1,List by its ID number all restaurants on 11th Street in Oakland.,SELECT id_restaurant FROM location WHERE city = 'oakland' AND street_name = '11th street',SELECT id_restaurant FROM location WHERE city = 'oakland' AND street_name = '11th street'
4,3476,video_games,1,List down the names of the games in the racing genre.,SELECT T1.game_name FROM game AS T1 INNER JOIN genre AS T2 ON T1.genre_id = T2.id WHERE T2.genre_name = 'Racing',SELECT T1.game_name FROM game AS T1 INNER JOIN genre AS T2 ON T1.genre_id = T2.id WHERE T2.genre_name = 'Racing'
5,8994,bike_share_1,0,Which is the station where no bike could not be borrowed form on the 2013/11/03 02:01:01? State the location of the station.,"SELECT T1.name, T1.lat, T1.long FROM station AS T1 INNER JOIN status AS T2 ON T1.id = T2.station_id WHERE T2.time = '3/11/2013 02:01:01' AND T2.bikes_available = 0","SELECT T1.name, T1.long FROM station AS T1 INNER JOIN status AS T2 ON T2.station_id = T1.id WHERE T2.time = '2013/11/03 02:01:01' AND T2.bikes_available = 0"
6,2946,professional_basketball,0,"In 2000, which team did the player who played the least minutes without missing a single game play in? Give the full name of the team.",SELECT T2.name FROM players_teams AS T1 INNER JOIN teams AS T2 ON T1.tmID = T2.tmID AND T1.year = T2.year WHERE T1.year = 2000 AND T1.GP = 82 ORDER BY T1.minutes ASC LIMIT 1,SELECT T1.tmID FROM teams AS T1 INNER JOIN players_teams AS T2 ON T1.tmID = T2.tmID AND T1.year = T2.year WHERE T2.GP = 82 AND T2.year = 2000 GROUP BY T1.tmID ORDER BY SUM(T2.PostMinutes) ASC LIMIT 1
7,3373,video_games,0,Provide the game publisher's name of the game with sales greater than 90% of the average sales in Japan.,SELECT DISTINCT p.publisher_name\nFROM region_sales rs\nJOIN region r ON rs.region_id = r.id\nJOIN game_platform gp ON rs.game_platform_id = gp.id\nJOIN game_publisher gpub ON gp.game_publisher_id = gpub.id\nJOIN publisher p ON gpub.publisher_id = p.id\nWHERE r.region_name = 'Japan'\n AND rs.num_sales > 0.9 * (\n SELECT AVG(rs.num_sales)\n FROM region_sales rs\n JOIN region r ON rs.region_id = r.id\n WHERE r.region_name = 'Japan'\n ),SELECT DISTINCT T5.publisher_name FROM region AS T1 INNER JOIN region_sales AS T2 ON T1.id = T2.region_id INNER JOIN game_platform AS T3 ON T2.game_platform_id = T3.id INNER JOIN game_publisher AS T4 ON T3.game_publisher_id = T4.id INNER JOIN publisher AS T5 ON T4.publisher_id = T5.id WHERE T2.num_sales * 10000000 > ( SELECT AVG(T2.num_sales) * 100000 * 90 FROM region AS T1 INNER JOIN region_sales AS T2 ON T1.id = T2.region_id WHERE T1.region_name = 'Japan' )
8,5585,shipping,0,What is the full name of the driver that has done the most shipmen

## Error Analysis — Sampled Predictions

Manual review of 5 correct and 5 incorrect predictions.

### Correct predictions
The model reliably handles standard patterns: multi-table joins (rows 0, 4),
aggregation with filtering (row 1), `ORDER BY ... LIMIT` for superlatives
(row 2), and simple filtered lookups (row 3). Minor cosmetic differences from
gold (e.g. join order, `COUNT(DISTINCT)` vs `COUNT`) don't affect the result
set, so they still score as matches.

### Incorrect predictions — breakdown

| id | db | Cause | Type |
|----|----|-------|------|
| 8994 | bike_share_1 | Wrong date format (`3/11/2013` vs `2013/11/03`) + extra `lat` column | Real miss |
| 2946 | professional_basketball | Different interpretation of "least minutes" (regular vs postseason) + wrong output column | Real miss (ambiguous) |
| 3373 | video_games | Model logic is sound; gold uses an odd `*10000000` scaling hack the model didn't replicate | Likely unfair to model |
| 5585 | shipping | Grouped by `driver_id` vs gold's `first_name, last_name` — diverges only on duplicate names | Real miss (edge case) |
| 9281 | movie_3 | Joined via customer's address instead of the **store's** address — answered wrong entity | Real miss |

### Takeaways
- **4 of 5 misses are genuine** (format, interpretation, grouping, join-path),
  so the 66% is mostly honest rather than inflated. One case (3373) penalizes a
  reasonable query for not matching a quirky gold, meaning true accuracy is
  marginally **higher** than 66%.
- Two recurring, fixable weaknesses:
  1. **Exact string/date literal formats** (id 8994).
  2. **Choosing the correct join path** when entities are nested, e.g.
     store-vs-customer address (id 9281).

In [ ]:
# moving the testing to dev set now
DEV_ROOT = "../Data/dev/dev_databases"          # adjust to your layout
dev = json.load(open("../Data/dev/dev.json"))
dev_dbs = {os.path.basename(p): glob.glob(os.path.join(p,"*.sqlite"))[0]
           for p in glob.glob(os.path.join(DEV_ROOT,"*")) if os.path.isdir(p)}

In [77]:
dev

[{'question_id': 0,
  'db_id': 'california_schools',
  'question': 'What is the highest eligible free rate for K-12 students in the schools in Alameda County?',
  'evidence': 'Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`',
  'SQL': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1",
  'difficulty': 'simple'},
 {'question_id': 1,
  'db_id': 'california_schools',
  'question': 'Please list the lowest three eligible free rates for students aged 5-17 in continuation schools.',
  'evidence': 'Eligible free rates for students aged 5-17 = `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)`',
  'SQL': "SELECT `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)` FROM frpm WHERE `Educational Option Type` = 'Continuation School' AND `Free Meal Count (Ages 5-17)` / `Enrollment (Ages 5-17)` IS NOT NULL ORDER BY `Free Meal Count 

In [79]:
len(dev), len(dev_dbs)

(1534, 11)

In [81]:
zr_table = pd.read_csv("zeroshot_table.csv")

In [84]:
import pandas as pd, json, sqlite3, glob, os, re
df = pd.read_csv("errors_ddl.csv")

# 1. THE KEY TEST — are gold and pred failing on the same questions?
ct = pd.crosstab(df.gold_err.notna(), df.pred_err.notna())
print(ct, "\n")
n = len(df)
p_g, p_p = df.gold_err.notna().mean(), df.pred_err.notna().mean()
print(f"both, observed: {ct.loc[True, True]}")
print(f"both, if independent: {p_g * p_p * n:.0f}")

# 2. where do broken golds live?
bad = df[df.gold_err.notna()]
print("\nbroken gold by db:\n", bad.groupby("db").size().sort_values(ascending=False))
print("\ngold error kinds:\n", bad.gold_err.str.slice(0, 45).value_counts().head(12))

pred_err  False  True 
gold_err              
False       766     33
True          4    197 

both, observed: 197
both, if independent: 46

broken gold by db:
 db
retail_world    199
shooting          2
dtype: int64

gold error kinds:
 gold_err
no such table: Order Details          45
no such column: T2.CompanyName        33
no such table: EmployeeTerritories    25
no such table: Territories            11
no such column: CompanyName           10
no such column: Salary                 5
no such column: Title                  5
no such column: Discontinued           4
no such column: UnitsInStock           4
no such column: T2.ShipCountry         4
no such column: T1.CompanyName         4
no such column: TitleOfCourtesy        3
Name: count, dtype: int64


In [85]:
DATA = "../Data"
dbs = {os.path.basename(p): glob.glob(os.path.join(p, "*.sqlite"))[0]
       for p in glob.glob(f"{DATA}/train/train_databases/*") if os.path.isdir(p)}

for db in bad.db.value_counts().head(3).index:
    con = sqlite3.connect(dbs[db])
    real = {r[0].lower() for r in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table'")}
    con.close()
    ref = set()
    for q in bad[bad.db == db].gold.head(30):
        ref |= {m.lower() for m in re.findall(
            r"(?:FROM|JOIN)\s+[`\"\[]?(\w+)", str(q), re.I)}
    print(f"\n{db}")
    print(f"  tables in file:      {sorted(real)}")
    print(f"  gold references:     {sorted(ref)}")
    print(f"  referenced but absent: {sorted(ref - real)}")


retail_world
  tables in file:      ['categories', 'customers', 'employees', 'orderdetails', 'orders', 'products', 'shippers', 'sqlite_sequence', 'suppliers']
  gold references:     ['employees', 'employeeterritories', 'order', 'orders', 'products', 'region', 'shippers', 'suppliers', 'territories']
  referenced but absent: ['employeeterritories', 'order', 'region', 'territories']

shooting
  tables in file:      ['incidents', 'officers', 'subjects']
  gold references:     ['incidents']
  referenced but absent: []
